In [ ]:
# v3 step 1: download MovieLens ml-32m once to Drive, then load it.
from pathlib import Path
import zipfile, urllib.request
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RAW_DIR = Path("/content/drive/MyDrive/tastelet/data/raw")  # change if your Drive folder differs
ZIP_PATH = RAW_DIR / "ml-32m.zip"
DATA_DIR = RAW_DIR / "ml-32m"

if not DATA_DIR.exists():  # download only the first time (~240 MB)
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-32m.zip", ZIP_PATH)
    zipfile.ZipFile(ZIP_PATH).extractall(RAW_DIR)

# Small number types: 32M rows would use ~2x the memory with pandas' defaults.
ratings = pd.read_csv(
    DATA_DIR / "ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"},
)
movies = pd.read_csv(DATA_DIR / "movies.csv")
print(f"{len(ratings):,} ratings · {ratings['userId'].nunique():,} users · {len(movies):,} movies")

Mounted at /content/drive
32,000,204 ratings · 200,948 users · 87,585 movies


In [ ]:
# v3 step 1: how many movies (and how many new ones) survive each "minimum ratings" cutoff?
CUTOFFS = [20, 50, 100]
NEW_FROM_YEAR = 2019

ratings_per_movie = ratings["movieId"].value_counts()  # movieId -> number of ratings
year = (
    movies.set_index("movieId")["title"]
    .str.extract(r"\((\d{4})\)\s*$")[0]  # "Toy Story (1995)" -> "1995"; no year -> NaN
    .astype("float")
)

rows = []
for cutoff in CUTOFFS:
    kept = ratings_per_movie[ratings_per_movie >= cutoff]
    kept_years = year.reindex(kept.index)
    rows.append({
        "min_ratings": cutoff,
        "movies_kept": len(kept),
        f"new_movies_{NEW_FROM_YEAR}+": int((kept_years >= NEW_FROM_YEAR).sum()),
        "share_of_ratings_kept": f"{kept.sum() / len(ratings):.1%}",
    })
summary = pd.DataFrame(rows)
summary

,min_ratings,movies_kept,new_movies_2019+,share_of_ratings_kept
0,20,23350,1886,99.1%
1,50,16034,1033,98.4%
2,100,12191,670,97.6%


In [ ]:
# v3 step 2: keep movies with >= 20 ratings, then split each user's ratings by time:
# oldest 80% -> train, next 10% -> validation, newest 10% -> test (no peeking at the future).
MIN_RATINGS = 20
TRAIN_SHARE, VAL_SHARE = 0.8, 0.1

kept_movies = ratings_per_movie[ratings_per_movie >= MIN_RATINGS].index
ratings_kept = ratings[ratings["movieId"].isin(kept_movies)]

# Position of each rating in its user's timeline, from 0 (oldest) to just under 1 (newest).
# Ties on timestamp break by movieId, so the split is the same on every run.
ratings_kept = ratings_kept.sort_values(["userId", "timestamp", "movieId"])
position = ratings_kept.groupby("userId").cumcount() / ratings_kept.groupby("userId")["movieId"].transform("size")

split = pd.Series("test", index=ratings_kept.index)
split[position < TRAIN_SHARE + VAL_SHARE] = "val"
split[position < TRAIN_SHARE] = "train"
ratings_kept = ratings_kept.assign(split=split)

print(ratings_kept["split"].value_counts(normalize=True).round(3))
by_split_year = ratings_kept.groupby("split")["timestamp"].median()
print(pd.to_datetime(by_split_year, unit="s").dt.date)  # test should be the most recent
print("users with no test rating:", (ratings_kept.groupby("userId")["split"].apply(lambda s: (s == "test").sum()) == 0).sum())

split
train    0.802
val      0.100
test     0.097
Name: proportion, dtype: float64
split
test     2012-09-15
train    2009-08-30
val      2011-07-24
Name: timestamp, dtype: object
users with no test rating: 3


In [ ]:
# v3 step 3: re-index ids, drop movies the model could never learn, save to Drive.
OUT_DIR = Path("/content/drive/MyDrive/tastelet/data/processed_v3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# A movie with no TRAIN rating gets an untrained (random) embedding, so it can't be scored.
train_movie_ids = set(ratings_kept.loc[ratings_kept["split"] == "train", "movieId"])
no_train = ratings_kept.loc[~ratings_kept["movieId"].isin(train_movie_ids), "movieId"].nunique()
print("movies with no train rating (dropped):", no_train)
ratings_final = ratings_kept[ratings_kept["movieId"].isin(train_movie_ids)]

# Same idea as v1: ids become 0..N-1 so they can index the embedding tables directly.
movie2idx = {int(m): i for i, m in enumerate(sorted(ratings_final["movieId"].unique()))}
user2idx = {int(u): i for i, u in enumerate(sorted(ratings_final["userId"].unique()))}
ratings_final = ratings_final.assign(
    movie_idx=ratings_final["movieId"].map(movie2idx).astype("int32"),
    user_idx=ratings_final["userId"].map(user2idx).astype("int32"),
)

# Parquet: about 5x smaller and 10x faster to load than CSV for 32M rows.
for name in ["train", "val", "test"]:
    part = ratings_final[ratings_final["split"] == name].drop(columns="split")
    part.to_parquet(OUT_DIR / f"ratings_{name}.parquet", index=False)
    print(name, f"{len(part):,} ratings")

import json
(OUT_DIR / "movie2idx.json").write_text(json.dumps(movie2idx))
(OUT_DIR / "user2idx.json").write_text(json.dumps(user2idx))
print(f"saved: {len(movie2idx):,} movies, {len(user2idx):,} users")

movies with no train rating (dropped): 6
train 25,458,081 ratings
val 3,180,816 ratings
test 3,086,867 ratings
saved: 23,344 movies, 200,948 users
